# Twitter Embeddings: Static vs Contextual

Pipeline used in this notebook:

**Tweets → NLTK Word Tokenization → Stop-word Removal → Cleaned Tweets → Embeddings**

### Static embeddings
- Word2Vec
- GloVe
- FastText

### Contextual embeddings
- BERT
- GPT-2


## 1. Install / Import Libraries


In [10]:
!pip install nltk gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB 2.0 MB/s eta 0:00:12
   ---------------------------------------- 0.1/24.4 MB 2.6 MB/s eta 0:00:10
   ---------------------------------------- 0.2/24.4 MB 2.3 MB/s eta 0:00:11
    --------------------------------------- 0.4/24.4 MB 2.2 MB/s eta 0:00:11
    --------------------------------------- 0.5/24.4 MB 2.2 MB/s eta 0:00:12
    --------------------------------------- 0.5/24.4 MB 2.4 MB/s eta 0:00:10
   - -------------------------------------- 0.6/24.4 MB 2.0 MB/s eta 0:00:12
   - -------------------------------------- 0.7/24.4 MB 1.9 MB/s eta 0:00:13
   - -------------------------------------- 0.7/24.4 MB 1.8 MB/s eta 0:00:13
   - -------------------------------------- 0.7/24.4 MB 1.6 MB/s eta 0:00:15
   - -------------------------------------- 0.8/24.4 MB 1.6 MB/s eta 0:00:16
   - -------------------------------------- 0.8/24.4 MB 1.5 MB/s eta 0:00:16
   - -


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Run this cell if the required packages are not installed.
# !pip install pandas numpy gensim nltk torch transformers scikit-learn

import re
import os
import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kk817\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kk817\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kk817\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. Example Tweets Dataset


In [12]:
tweets = [
    "I love this phone, the battery life is amazing!",
    "The phone battery died after two hours. Very bad.",
    "Apple launched a new iPhone today.",
    "I ate an apple after lunch.",
    "The movie was amazing and the acting was excellent.",
    "This movie is terrible, I want my money back.",
    "Python is great for machine learning.",
    "I am learning Python programming for my project.",
    "The bank approved my loan yesterday.",
    "I went to the river bank to take pictures.",
    "AI is changing the future of software engineering.",
    "The software update fixed many security bugs.",
    "Cybersecurity is important for every company.",
    "Hackers attacked the company's server last night.",
    "This laptop is fast but the keyboard is uncomfortable.",
    "The new model understands the tweet context very well."
]

df = pd.DataFrame({'tweet': tweets})
df.head(20)


,tweet
0,"I love this phone, the battery life is amazing!"
1,The phone battery died after two hours. Very bad.
2,Apple launched a new iPhone today.
3,I ate an apple after lunch.
4,The movie was amazing and the acting was excel...
5,"This movie is terrible, I want my money back."
6,Python is great for machine learning.
7,I am learning Python programming for my project.
8,The bank approved my loan yesterday.
9,I went to the river bank to take pictures.


## 3. Text Preprocessing with NLTK

The preprocessing phase performs:

1. Lowercasing
2. Removing URLs
3. Removing mentions (`@user`)
4. Removing the `#` symbol while keeping the hashtag word
5. Word tokenization using NLTK
6. Removing punctuation
7. Removing English stop words

Example:

`The movie was amazing and the acting was excellent!`

becomes approximately:

`['movie', 'amazing', 'acting', 'excellent']`


In [13]:
stop_words = set(stopwords.words('english'))

def preprocess_tweet(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\\w+', '', text)

    # Keep hashtag text but remove '#'
    text = re.sub(r'#', '', text)

    # Word tokenization using NLTK
    tokens = word_tokenize(text)

    # Keep alphabetic words and remove stop words
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return tokens

df['tokens'] = df['tweet'].apply(preprocess_tweet)

df[['tweet', 'tokens']]


,tweet,tokens
0,"I love this phone, the battery life is amazing!","[love, phone, battery, life, amazing]"
1,The phone battery died after two hours. Very bad.,"[phone, battery, died, two, hours, bad]"
2,Apple launched a new iPhone today.,"[apple, launched, new, iphone, today]"
3,I ate an apple after lunch.,"[ate, apple, lunch]"
4,The movie was amazing and the acting was excel...,"[movie, amazing, acting, excellent]"
5,"This movie is terrible, I want my money back.","[movie, terrible, want, money, back]"
6,Python is great for machine learning.,"[python, great, machine, learning]"
7,I am learning Python programming for my project.,"[learning, python, programming, project]"
8,The bank approved my loan yesterday.,"[bank, approved, loan, yesterday]"
9,I went to the river bank to take pictures.,"[went, river, bank, take, pictures]"


## 4. Inspect Original vs Cleaned Tweets


In [14]:
for original, tokens in zip(df['tweet'], df['tokens']):
    print('Original :', original)
    print('Cleaned  :', tokens)
    print('-' * 70)


Original : I love this phone, the battery life is amazing!
Cleaned  : ['love', 'phone', 'battery', 'life', 'amazing']
----------------------------------------------------------------------
Original : The phone battery died after two hours. Very bad.
Cleaned  : ['phone', 'battery', 'died', 'two', 'hours', 'bad']
----------------------------------------------------------------------
Original : Apple launched a new iPhone today.
Cleaned  : ['apple', 'launched', 'new', 'iphone', 'today']
----------------------------------------------------------------------
Original : I ate an apple after lunch.
Cleaned  : ['ate', 'apple', 'lunch']
----------------------------------------------------------------------
Original : The movie was amazing and the acting was excellent.
Cleaned  : ['movie', 'amazing', 'acting', 'excellent']
----------------------------------------------------------------------
Original : This movie is terrible, I want my money back.
Cleaned  : ['movie', 'terrible', 'want', 'money

## 5. Prepare Corpus for Embeddings

Word2Vec and FastText expect a corpus containing tokenized sentences.


In [15]:
tokenized_tweets = df['tokens'].tolist()

print('Number of tweets:', len(tokenized_tweets))
print('First cleaned tweet:', tokenized_tweets[0])


Number of tweets: 16
First cleaned tweet: ['love', 'phone', 'battery', 'life', 'amazing']


## 6. Static Embedding — Word2Vec

Word2Vec learns a fixed vector for each word from the cleaned tweet corpus.

**Important:** the word `bank` has one Word2Vec vector regardless of whether it appears in a financial or river context.


In [16]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

word = 'bank'
vector = w2v_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

print('\nSimilar words:')
print(w2v_model.wv.most_similar(word, topn=5))


Word: bank
Vector shape: (100,)
First 10 values: [ 0.00038309  0.0032622  -0.00728072 -0.00151712  0.00731176  0.00553623
 -0.00336169  0.00454942 -0.00868977  0.00471174]

Similar words:
[('yesterday', 0.24527476727962494), ('well', 0.23155668377876282), ('amazing', 0.22988075017929077), ('excellent', 0.22535552084445953), ('fast', 0.20152848958969116)]


## 7. Static Embedding — GloVe

GloVe is also a static embedding technique. A word has one fixed vector.

Download a pretrained GloVe file such as `glove.6B.100d.txt` and place it beside this notebook.


In [17]:
def load_glove(glove_path):
    embeddings = {}

    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype=np.float32)
            embeddings[word] = vector

    return embeddings

glove_path = 'glove.6B.100d.txt'

if os.path.exists(glove_path):
    glove = load_glove(glove_path)
    word = 'bank'
    print('Word:', word)
    print('Vector shape:', glove[word].shape)
    print('First 10 values:', glove[word][:10])
else:
    print('GloVe file not found.')
    print('Place glove.6B.100d.txt in the notebook directory to run this section.')


GloVe file not found.
Place glove.6B.100d.txt in the notebook directory to run this section.


## 8. Static Embedding — FastText

FastText is similar to Word2Vec but represents words using character n-grams. This is particularly useful for tweets because tweets often contain misspellings, abbreviations, hashtags and rare words.


In [18]:
from gensim.models import FastText

fasttext_model = FastText(
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

fasttext_model.build_vocab(corpus_iterable=tokenized_tweets)
fasttext_model.train(
    corpus_iterable=tokenized_tweets,
    total_examples=len(tokenized_tweets),
    epochs=100
)

word = 'bank'
vector = fasttext_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

unknown_word = 'amazzzzing'
unknown_vector = fasttext_model.wv[unknown_word]
print('\nFastText can generate a vector for:', unknown_word)
print('Vector shape:', unknown_vector.shape)


Word: bank
Vector shape: (100,)
First 10 values: [-3.7789496e-03 -2.8636651e-03 -2.1460950e-03 -1.8785144e-03
  1.7680831e-03 -7.3039155e-05  1.1883718e-03 -1.8888668e-03
  9.4088893e-05 -2.4465250e-03]

FastText can generate a vector for: amazzzzing
Vector shape: (100,)


## 9. Contextual Embedding — BERT

BERT creates contextual token representations.

Compare:
- `I deposited money in the bank.`
- `I sat near the river bank.`

The representation of `bank` changes because its surrounding context changes.

**Note:** We use the original text here rather than the stop-word-removed text. Removing words such as `in`, `the`, and `near` would remove useful context for a pretrained contextual model.


In [19]:
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

contextual_tweets = [
    'I deposited money in the bank.',
    'I sat near the river bank.'
]

encoded = bert_tokenizer(
    contextual_tweets,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    outputs = bert_model(**encoded)

token_embeddings = outputs.last_hidden_state

print('Token embedding shape:', token_embeddings.shape)
print('BERT hidden size:', token_embeddings.shape[-1])


C:\Users\kk817\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2111.82it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loa

Token embedding shape: torch.Size([2, 9, 768])
BERT hidden size: 768


## 10. BERT — Tweet-Level Embedding with Mean Pooling


In [20]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

bert_tweet_embeddings = mean_pool(
    token_embeddings,
    encoded['attention_mask']
)

print('Tweet embedding shape:', bert_tweet_embeddings.shape)
print('First tweet embedding, first 10 values:')
print(bert_tweet_embeddings[0][:10].numpy())


Tweet embedding shape: torch.Size([2, 768])
First tweet embedding, first 10 values:
[ 0.4423436  -0.15134196 -0.13635205  0.14299276  0.25344327 -0.04802983
 -0.19784291  0.65312076  0.23939723 -0.55328757]


## 11. Contextual Embedding — GPT-2

GPT-2 is a causal language model. Its hidden states can also be used as contextual representations.

**Important:** GPT-2 is not specifically optimized as a sentence-embedding model. This section demonstrates the concept of contextual representations rather than claiming GPT-2 is the best choice for semantic embeddings.


In [ ]:
from transformers import AutoTokenizer, AutoModel

gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gpt_model = AutoModel.from_pretrained('gpt2')
gpt_model.eval()

gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_encoded = gpt_tokenizer(
    contextual_tweets,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    gpt_outputs = gpt_model(**gpt_encoded)

gpt_token_embeddings = gpt_outputs.last_hidden_state

gpt_tweet_embeddings = mean_pool(
    gpt_token_embeddings,
    gpt_encoded['attention_mask']
)

print('First tweet embedding, first 10 values:')
print(gpt_tweet_embeddings[0][:10].numpy())

print('GPT-2 token embedding shape:', gpt_token_embeddings.shape)
print('GPT-2 tweet embedding shape:', gpt_tweet_embeddings.shape)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9247.91it/s]

First tweet embedding, first 10 values:
[ 2.5560176e-01 -4.7513461e-03 -3.4219432e-01 -2.0040102e-01
  1.2607931e-01 -2.1061845e-01  6.0537190e+00  1.7905077e-01
 -4.5304996e-01 -4.5382433e-02]
GPT-2 token embedding shape: torch.Size([2, 7, 768])
GPT-2 tweet embedding shape: torch.Size([2, 768])


: 

## 12. Static vs Contextual Embeddings

| Method | Type | Main idea |
|---|---|---|
| Word2Vec | Static | One vector per word learned from local context |
| GloVe | Static | One vector per word learned from global co-occurrence |
| FastText | Static | Word vectors using character n-grams |
| BERT | Contextual | Token representation depends on both sides of context |
| GPT-2 | Contextual | Token representation depends on preceding context |

### Example: `bank`

**Static:**
`bank → fixed vector`

**Contextual:**
`financial context + bank → vector A`

`river context + bank → vector B`


## 13. Final Pipeline

```text
Raw Tweets
    ↓
Lowercase / URL / Mention Cleaning
    ↓
NLTK Word Tokenization
    ↓
Stop-word Removal
    ↓
Clean Tokenized Tweets
    ↓
 ┌───────────────┬────────────────┐
 │ Static        │ Contextual     │
 │               │                │
 │ Word2Vec      │ BERT           │
 │ GloVe         │ GPT-2          │
 │ FastText      │                │
 └───────────────┴────────────────┘
```

**Why BERT/GPT use the original text:** stop-word removal is appropriate for traditional preprocessing, but contextual transformer models were pretrained on natural language where function words and sentence structure carry meaning. Therefore, preprocessing for transformers should generally be much lighter.
